# 🧠 Robust Multimodal Sentiment Analysis
## Combatting Text Fragility via Adaptive Modality Balancing

**3-Stage Training Pipeline** with:
- Adaptive Curriculum Dropout (Novel)
- Corrected OGM-GE Gradient Modulation
- Inter-Modal Contrastive Alignment (InfoNCE)
- Learnable Gating Fusion

---
### Quick Start
1. Run **Section 1** (Setup) first
2. Run **Section 2** (Data) to download the dataset
3. Run **Sections 3-5** for training (Stage 1 → 2 → 3)
4. Run **Section 6** for ablation study
5. Run **Section 7** for robustness evaluation (KEY results)
6. Run **Section 8** for visualizations

## 1. 🔧 Environment Setup

In [2]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. Go to Runtime > Change runtime type > GPU")


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


In [5]:
# Clone the repository from GitHub
!git clone https://github.com/saimayithri/Multi-Modal-Sentiment-Analysis.git
%cd Multi-Modal-Sentiment-Analysis

import os
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

Cloning into 'Multi-Modal-Sentiment-Analysis'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 36 (delta 7), reused 36 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 36.37 KiB | 2.80 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/Multi-Modal-Sentiment-Analysis
Working directory: /content/Multi-Modal-Sentiment-Analysis
Files: ['MSA_Research_Colab.ipynb', 'requirements.txt', '.gitignore', 'datasets', 'modules', 'models', 'src', '.git', 'run']


## 2. 📦 Dataset Download

In [11]:
from google.colab import drive
import os
import shutil

# 1. Mount your Google Drive
drive.mount('/content/drive')

os.makedirs('data', exist_ok=True)

# 2. Since the folder is in "Shared with me", the easiest way to access it in Colab
# is to go to Google Drive, right-click the "Processed" folder, and click "Add shortcut to Drive".
# Assuming you added the shortcut to your main Drive, the path will be:
drive_folder = '/content/drive/MyDrive/My_MSA_Project/data/' # Corrected path to include /content/

print(f"\nLooking for files in: {drive_folder}")

try:
    # List contents of the drive folder for debugging
    if os.path.exists(drive_folder):
        print(f"Contents of {drive_folder}:")
        for item in os.listdir(drive_folder):
            print(f"  - {item}")
    else:
        print(f"⚠️ {drive_folder} does not exist. Please check the path and ensure the shortcut is created.")

    # Copy Unaligned Data and rename it for the next cell
    source_file = os.path.join(drive_folder, 'unaligned_50.pkl')
    destination_file = 'data/mosi_data.pkl'

    if os.path.exists(source_file):
        print(f"Copying {os.path.basename(source_file)} to {destination_file}...")
        shutil.copy(source_file, destination_file)
        print("✅ Data ready!")
    else:
        print(f"⚠️ {os.path.basename(source_file)} not found in {drive_folder}. Please ensure the file exists.")

except Exception as e:
    print(f"Error during file copy: {e}")

print("\nFiles in your local data/ folder:")
for f in os.listdir('data'):
    print(f"  {f}: {os.path.getsize(os.path.join('data', f)) / 1e6:.1f} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Looking for files in: /content/drive/MyDrive/My_MSA_Project/data/
Contents of /content/drive/MyDrive/My_MSA_Project/data/:
  - unaligned_50.pkl
Copying unaligned_50.pkl to data/mosi_data.pkl...
✅ Data ready!

Files in your local data/ folder:
  mosi_data.pkl: 554.2 MB


In [13]:
# Verify dataset structure
import pickle
import numpy as np

with open('data/mosi_data.pkl', 'rb') as f:
    data = pickle.load(f)

for split in ['train', 'valid', 'test']:
    print(f"\n--- {split.upper()} ---")
    for key in data[split]:
        arr = data[split][key]
        # Check if the array is a numpy array before accessing .shape and .dtype
        if isinstance(arr, np.ndarray):
            print(f"  {key:20s}: shape={str(arr.shape):20s} dtype={arr.dtype}")
        else:
            # For lists or other types, print type and length
            print(f"  {key:20s}: type={type(arr).__name__:20s} len={len(arr)}")

# These lines assume 'text', 'audio', and 'vision' are numpy arrays
print(f"\nText dim: {data['train']['text'].shape[2]}")
print(f"Audio dim: {data['train']['audio'].shape[2]}")
print(f"Vision dim: {data['train']['vision'].shape[2]}")
print(f"Train samples: {len(data['train']['text'])}")
print(f"Valid samples: {len(data['valid']['text'])}")
print(f"Test samples: {len(data['test']['text'])}")


--- TRAIN ---
  raw_text            : shape=(1284,)              dtype=<U581
  audio               : shape=(1284, 375, 5)       dtype=float64
  vision              : shape=(1284, 500, 20)      dtype=float64
  id                  : type=list                 len=1284
  text                : shape=(1284, 50, 768)      dtype=float32
  text_bert           : shape=(1284, 3, 50)        dtype=int64
  audio_lengths       : type=list                 len=1284
  vision_lengths      : type=list                 len=1284
  annotations         : type=list                 len=1284
  classification_labels: shape=(1284,)              dtype=float64
  regression_labels   : shape=(1284,)              dtype=float64

--- VALID ---
  raw_text            : shape=(229,)               dtype=<U231
  audio               : shape=(229, 375, 5)        dtype=float64
  vision              : shape=(229, 500, 20)       dtype=float64
  id                  : type=list                 len=229
  text                : shape=(

## 3. 🎓 Stage 1: Train Text-Only Teacher

In [22]:
# Install required packages
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: Operation cancelled by user


In [17]:
import sys
import os

# Add the current directory (root of the cloned repo) to the Python path
# This is necessary for importing modules like 'datasets.dataloader'
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())
print(f"Current working directory '{os.getcwd()}' added to sys.path.")

Current working directory '/content/Multi-Modal-Sentiment-Analysis' added to sys.path.


In [19]:
os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)

# Set PYTHONPATH to include the current directory so modules like 'datasets' can be found
!PYTHONPATH=. python run/train.py \
    --stage 1 \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --num_epochs 40 \
    --batch_size 32 \
    --lr 1e-4 \
    --seed 42 \
    --log_dir logs

--- Starting Stage 1: Training Text-Only Teacher ---
Stage 1 | Epoch  1 | Loss: 1.9014 | Valid Acc-2: 0.5972 | F1: 0.4884
*** New Best Teacher! Saving to models/teacher_text_best.pt ***
Stage 1 | Epoch  2 | Loss: 1.8384 | Valid Acc-2: 0.5741 | F1: 0.4187
Stage 1 | Epoch  3 | Loss: 1.8051 | Valid Acc-2: 0.5741 | F1: 0.4187
Stage 1 | Epoch  4 | Loss: 1.7557 | Valid Acc-2: 0.6250 | F1: 0.5759
*** New Best Teacher! Saving to models/teacher_text_best.pt ***
Stage 1 | Epoch  5 | Loss: 1.6972 | Valid Acc-2: 0.7222 | F1: 0.7166
*** New Best Teacher! Saving to models/teacher_text_best.pt ***
Stage 1 | Epoch  6 | Loss: 1.6322 | Valid Acc-2: 0.7731 | F1: 0.7744
*** New Best Teacher! Saving to models/teacher_text_best.pt ***
Stage 1 | Epoch  7 | Loss: 1.5587 | Valid Acc-2: 0.7731 | F1: 0.7744
Stage 1 | Epoch  8 | Loss: 1.5233 | Valid Acc-2: 0.7824 | F1: 0.7834
*** New Best Teacher! Saving to models/teacher_text_best.pt ***
Stage 1 | Epoch  9 | Loss: 1.4960 | Valid Acc-2: 0.7593 | F1: 0.7596
Stage 

## 4. 📡 Stage 2: Train Audio/Vision Students (Knowledge Distillation)

In [23]:
!git pull origin main


remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 623 bytes | 311.00 KiB/s, done.
From https://github.com/saimayithri/Multi-Modal-Sentiment-Analysis
 * branch            main       -> FETCH_HEAD
   aad1cfc..a149df5  main       -> origin/main
Updating aad1cfc..a149df5
Fast-forward
 run/train.py | 15 ++++++++++++---
 1 file changed, 12 insertions(+), 3 deletions(-)


In [24]:
!PYTHONPATH=. python run/train.py \
    --stage 2 \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --num_epochs 40 \
    --batch_size 32 \
    --lr 1e-4 \
    --seed 42 \
    --log_dir logs

--- Starting Stage 2: Training A/V encoders with Decoupled Updates ---
Teacher model loaded and frozen.
Stage 2 | Epoch  1 | A-Acc: 0.4259 | V-Acc: 0.4537 | Avg: 0.4398
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  2 | A-Acc: 0.4676 | V-Acc: 0.5648 | Avg: 0.5162
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  3 | A-Acc: 0.4861 | V-Acc: 0.5694 | Avg: 0.5278
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  4 | A-Acc: 0.5278 | V-Acc: 0.5741 | Avg: 0.5509
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  5 | A-Acc: 0.5741 | V-Acc: 0.5741 | Avg: 0.5741
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  6 | A-Acc: 0.5926 | V-Acc: 0.5741 | Avg: 0.5833
*** New Best A/V Students! Saving to models/students_distilled_aligned.pt ***
Stage 2 | Epoch  7 | A-Acc: 0.5741 | V

## 5. 🔥 Stage 3: Full Model Fine-tuning (All Novel Techniques)

In [ ]:
# Full model with ALL novel techniques enabled
!python run/train.py \
    --stage 3 \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --modality_dropout 0.15 \
    --use_adaptive_dropout \
    --use_ogm \
    --use_contrastive \
    --num_epochs 40 \
    --batch_size 32 \
    --seed 42 \
    --log_dir logs

### 5b. Train a Text-Dominant Baseline (for comparison)
This trains Stage 3 **without** any of our techniques — creating a text-dominant model to compare against.

In [ ]:
# Baseline: no dropout, no OGM, no contrastive
!python run/train.py \
    --stage 3 \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --final_model_path models/baseline_text_dominant.pt \
    --num_epochs 40 \
    --batch_size 32 \
    --seed 42 \
    --log_dir logs

## 6. 🔬 Ablation Study (Modality Analysis)

In [ ]:
!python run/ablation_study.py \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --final_model_path models/robust_hybrid_best.pt \
    --student_model_path models/students_distilled_aligned.pt

## 7. 🛡️ Robustness Evaluation (KEY RESULTS)
This is the **most important** section. It produces the evidence that our balanced model
is more robust than text-dominant models when text is corrupted.

In [ ]:
# Test OUR model (balanced) under text corruption
!python run/robustness_test.py \
    --model_path models/robust_hybrid_best.pt \
    --model_name "Ours (Balanced)" \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --noise_types gaussian token_dropout \
    --output_dir figures

In [ ]:
# Test BASELINE model (text-dominant) under same corruption
!python run/robustness_test.py \
    --model_path models/baseline_text_dominant.pt \
    --model_name "Baseline (Text-Dominant)" \
    --data_path data/mosi_data.pkl \
    --dataset mosi \
    --noise_types gaussian token_dropout \
    --output_dir figures

In [ ]:
# Compare robustness side-by-side
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load results (you may need to load from separate files)
results_path = 'logs/robustness_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("\n📊 Robustness Results Loaded")
    print(f"  Model: {results['model_name']}")
    print(f"  Text-Off Acc-2: {results['text_off_metrics']['acc2']:.4f}")
    print(f"  Text-Off F1:    {results['text_off_metrics']['f1']:.4f}")
else:
    print("Run robustness tests first!")

### 7b. Combined Robustness Comparison Plot

In [ ]:
# Side-by-side robustness comparison (run after both models are tested)
import sys
sys.path.append('.')
from run.robustness_test import (
    evaluate_with_corruption, add_gaussian_noise, token_dropout
)
from datasets.dataloader import getdataloader
from models.msamodel import MSAModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataloader, orig_dim = getdataloader('mosi', 32, 'data/mosi_data.pkl')

# Load both models
models = {}
for name, path in [('Ours (Balanced)', 'models/robust_hybrid_best.pt'),
                    ('Baseline (Text-Dominant)', 'models/baseline_text_dominant.pt')]:
    if os.path.exists(path):
        m = MSAModel(output_dim=7, orig_dim=orig_dim, proj_dim=40, layers=5).to(device)
        m.load_state_dict(torch.load(path, map_location=device), strict=False)
        m.eval()
        models[name] = m
        print(f"Loaded: {name}")

if len(models) == 2:
    levels = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors = {'Ours (Balanced)': '#2ECC71', 'Baseline (Text-Dominant)': '#E74C3C'}

    for noise_idx, (noise_name, noise_fn, param_name) in enumerate([
        ('Gaussian Noise', add_gaussian_noise, 'noise_ratio'),
        ('Token Dropout', token_dropout, 'drop_ratio')
    ]):
        ax = axes[noise_idx]
        for model_name, model in models.items():
            accs = []
            for level in levels:
                if level == 0:
                    m = evaluate_with_corruption(model, dataloader['test'], device)
                else:
                    m = evaluate_with_corruption(
                        model, dataloader['test'], device,
                        noise_fn, {param_name: level})
                accs.append(m['acc2'])
            ax.plot(levels, accs, '-o', color=colors[model_name],
                    linewidth=2.5, markersize=7, label=model_name)

        ax.set_title(f'Accuracy Under {noise_name}', fontweight='bold', fontsize=14)
        ax.set_xlabel('Corruption Level', fontsize=12)
        ax.set_ylabel('Binary Accuracy (Acc-2)', fontsize=12)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-0.05, 1.05)

    plt.tight_layout()
    plt.savefig('figures/robustness_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\n✅ This is the KEY figure for your paper!")
    print("A flatter green line = our model is more robust.")
else:
    print("Need both models trained. Run Stage 3 and the baseline cell above.")

## 8. 📊 Visualization Suite

In [ ]:
# Generate all training visualization plots
os.makedirs('figures', exist_ok=True)
!python run/visualize.py --log_dir logs --output_dir figures --stage 3 --seed 42

In [ ]:
# Display all generated figures inline
from IPython.display import Image, display
import glob

figures = sorted(glob.glob('figures/*.png'))
for fig_path in figures:
    print(f"\n{'='*60}")
    print(f"  {os.path.basename(fig_path)}")
    print(f"{'='*60}")
    display(Image(filename=fig_path, width=800))

In [ ]:
# t-SNE visualization (takes ~2 minutes)
!python run/visualize.py \
    --log_dir logs --output_dir figures \
    --tsne \
    --model_path models/robust_hybrid_best.pt \
    --data_path data/mosi_data.pkl

In [ ]:
if os.path.exists('figures/tsne_representations.png'):
    display(Image(filename='figures/tsne_representations.png', width=900))

## 9. 📋 Final Results Summary Table

In [ ]:
import pandas as pd
import json

# Compile all results into a publication-ready table
print("\n" + "="*80)
print("  FINAL RESULTS SUMMARY")
print("="*80)

# Load Stage 3 log
log_path = 'logs/training_log_stage3_seed42.json'
if os.path.exists(log_path):
    with open(log_path) as f:
        log = json.load(f)

    # Best epoch metrics
    best_epoch = max(log['epochs'], key=lambda e: e.get('val_acc2', 0))

    print(f"\n📈 Best Model (Epoch {best_epoch['epoch']}):")
    print(f"  Acc-2: {best_epoch.get('val_acc2', 'N/A')}")
    print(f"  Acc-7: {best_epoch.get('val_acc7', 'N/A')}")
    print(f"  F1:    {best_epoch.get('val_f1', 'N/A')}")
    print(f"  MAE:   {best_epoch.get('val_mae', 'N/A')}")
    print(f"  Corr:  {best_epoch.get('val_corr', 'N/A')}")

    # Final gate distribution
    final = log['epochs'][-1]
    if 'avg_gate_text' in final:
        print(f"\n⚖️ Final Gate Distribution:")
        print(f"  Text:   {final['avg_gate_text']:.4f}")
        print(f"  Audio:  {final['avg_gate_audio']:.4f}")
        print(f"  Vision: {final['avg_gate_vision']:.4f}")
        uniform = abs(final['avg_gate_text'] - 1/3) + abs(final['avg_gate_audio'] - 1/3) + abs(final['avg_gate_vision'] - 1/3)
        print(f"  Deviation from uniform: {uniform:.4f} (lower = more balanced)")
else:
    print("Run training first to see results.")

In [ ]:
# Generate LaTeX-ready results table for your paper
print("\n📝 LaTeX Table (copy-paste into your paper):")
print("")
print(r"\begin{table}[h]")
print(r"\centering")
print(r"\caption{Performance comparison under text corruption on CMU-MOSI}")
print(r"\begin{tabular}{lcccc}")
print(r"\toprule")
print(r"Method & Clean Acc-2 & Noisy-30\% & Noisy-50\% & Text-Off \\")
print(r"\midrule")
print(r"Text-Dominant Baseline & -- & -- & -- & -- \\")
print(r"Ours (Balanced) & -- & -- & -- & -- \\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\label{tab:robustness}")
print(r"\end{table}")
print("\n(Fill in the -- values from your robustness test results above)")

## 10. 💾 Save Everything to Google Drive

In [ ]:
# Mount Google Drive and save results
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/MSA_Results'
os.makedirs(save_dir, exist_ok=True)

# Save models
for f in ['models/robust_hybrid_best.pt', 'models/teacher_text_best.pt',
          'models/students_distilled_aligned.pt', 'models/baseline_text_dominant.pt']:
    if os.path.exists(f):
        shutil.copy(f, save_dir)
        print(f"Saved: {f}")

# Save logs and figures
if os.path.exists('logs'):
    shutil.copytree('logs', os.path.join(save_dir, 'logs'), dirs_exist_ok=True)
if os.path.exists('figures'):
    shutil.copytree('figures', os.path.join(save_dir, 'figures'), dirs_exist_ok=True)

print(f"\n✅ All results saved to: {save_dir}")